## Set EM algorithm parameters, load packages, and load the dataset 

In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__)), "em-master"))  # Optional: activates local environment

# Optional: define data and code directory relative to current script
script_dir = dirname(dirname(@__DIR__))                      
em_dir = joinpath(script_dir, "em-master")  # EM algorithm folder
study_file = joinpath(script_dir, "Data", "Processed", "Study2.csv") #change to study1.csv for study1


# Set model and EM algorithm parameters
full = false               # Use diagonal covariance matrix if 'false'
emtol = 1e-3               # EM stopping threshold based on relative change
push!(LOAD_PATH, em_dir)  # Add directory for accessing custom files

# Load required packages
using EM, Distributions, LinearAlgebra, Optim, Statistics, Random
using GLM, DataFrames, NamedArrays, SpecialFunctions, CSV

# Load dataset
data = CSV.read(study_file, DataFrame)

# Rename 'subn' to 'sub'
rename!(data, :subn => :sub)

# Identify unique subjects and create a mapping to sequential IDs
unique_subjects = unique(data.sub)
subject_map = Dict(unique_subjects[i] => i for i in 1:length(unique_subjects))

# Apply the mapping to make 'sub' sequential as this is needed for EM
data.sub .= [subject_map[sub] for sub in data.sub]
# show data
data


Row,Trial,Choice,Reward,TrialType,FeedbackType,Reaction,Condition,Condition.Type,sub,C1,C2
,Int64,Int64,Int64,Bool,Int64,Int64,Int64,String7,Int64,Int64,Int64
1,1,1,10,false,0,1188,2,gain,1,1,0
2,2,1,-10,false,1,1968,4,loss,1,1,0
3,3,2,-10,true,1,1376,3,loss,1,0,1
4,4,1,-10,true,1,1998,4,loss,1,1,0
5,5,1,-1,true,1,1335,3,loss,1,1,0
6,6,2,1,false,1,2254,2,gain,1,0,1
7,7,2,1,false,1,2045,2,gain,1,0,1
8,8,2,-10,true,1,2725,4,loss,1,0,1
9,9,2,10,false,0,2377,1,gain,1,0,1


## Model M4 function
## This model has two learning rates for true and two for false 


In [ ]:
# EM variables
NS = 57                       # Number of subjects (set to 47 for study 1 and 57 for study 2)
subs = 1:NS                   # Subject indices
X = ones(Int, NS)             # design matrix specifying the group level model
startbetas = [0 0 0 0 0]      # starting points for group level parameters
startsigma = [1, 1, 1, 1, 1]     # Initial sigma(variance) values

# Model function to calculate negative log-likelihood
function M4(params, data)
    
    # (this is unit normal cdf,so squashes any input to 0-1 and 
  # transforms unit normal prior to uniform prior)
    lr_params = @. 1 / (1 + exp(-params[1:4]))
    Conf_True, Conf_False, Disconf_True, Disconf_False = lr_params
    
    # range 0-50
    Beta = 25 * (1 + erf(params[5] / sqrt(2)))
    
    # Extract data columns (using multiple assignment)
    R, C1, C2, TrialType, FeedbackType, Condition = (
        data.Reward, data.C1, data.C2, data.TrialType, 
        data.FeedbackType, data.Condition
    )
    
    # Initialize Q-values matrix
    Q = zeros(typeof(Beta), 2, 4)
    lik = 0.0
    
    for t in eachindex(C1)
        QC = Condition[t]
        chosen = C1[t] == 1 ? 1 : 2
        unchosen = 3 - chosen
        
        # Update likelihood based on choice probability
        lik += -log1p(exp(-Beta * (Q[chosen, QC] - Q[unchosen, QC])))
        
        # Calculate prediction error
        is_chosen_feedback = FeedbackType[t] == 1
        pe = is_chosen_feedback ? 
             (R[t] - Q[chosen, QC]) : 
             (R[t] - Q[unchosen, QC])
        
        # Determine learning rates based on trial type and prediction error
        is_true_trial = TrialType[t] == 1
        is_positive_pe = pe > 0
        
        # Select appropriate learning rate based on conditions
        function get_learning_rate(is_factual)
            if is_true_trial
                if is_factual == is_positive_pe
                    return Conf_True
                else
                    return Disconf_True
                end
            else
                if is_factual == is_positive_pe
                    return Conf_False
                else
                    return Disconf_False
                end
            end
        end
        
        # Update Q-values separately for chosen and unchosen options
        options = [(1, chosen), (2, unchosen)]  # Pairs of (index, option)
        
        for (option_index, current_option) in options
            # For chosen feedback (1): update chosen option (index 1)
            # For unchosen feedback (0): update unchosen option (index 2)
            is_correct_option_to_update = (option_index == 1 && is_chosen_feedback) || 
                                        (option_index == 2 && !is_chosen_feedback)
            
            if is_correct_option_to_update
                lr = get_learning_rate(is_chosen_feedback)
                Q[current_option, QC] = (1 - lr) * Q[current_option, QC] + lr * R[t]
            end
        end
    end
    
    return -lik
end


## Run em to get best fit parameters for each subject


In [ ]:
# run em
# x contains the parameters for each subject (note not the same as variable X)
# l and h are per-subject likelihood and hessians

(betas, sigma, x, l, h) = em(data, subs, X, startbetas, startsigma, M4, maxiter=2000, emtol=emtol, full=full)


## Generate Model Statistics

## (LOOCV)




In [ ]:
using CSV
using DataFrames
#compute unbiased per subject marginal likelihoods via cross validation.

liks = loocv(data,subs,x,X,betas,sigma,M4;maxiter=2000,emtol=emtol, full=full)
sum(liks)

# Convert the Vector to a DataFrame
liks_df = DataFrame(Values = liks)



## Write the loocv scores to a CSV file

In [ ]:

CSV.write("M4LOOCV.csv", liks_df)

## The following function transforms back the parameters to their original value and is used to for calculating individual points, the mean, standard error of the mean, and confidence interval. 

In [ ]:
using SpecialFunctions  # Required for the erf function
using DataFrames
using CSV
using LinearAlgebra  # For Matrix conversion
using Statistics  # For std function

(standarderrors, pvalues, covmtx) = emerrors(data, subs, x, X, h, betas, sigma, M4)
se = reshape(standarderrors, 1, length(betas))

function transform_back(x::Matrix)
    # Extract parameters directly from columns
    params1 = x[:, 1]
    params2 = x[:, 2]
    params3 = x[:, 3]
    params4 = x[:, 4]
    params5 = x[:, 5]
    
    # Transform parameters using exp function
    Conf_True = 1.0 ./ (1.0 .+ exp.(-params1))
    Conf_False = 1.0 ./ (1.0 .+ exp.(-params2))
    Disconf_True = 1.0 ./ (1.0 .+ exp.(-params3))
    Disconf_False = 1.0 ./ (1.0 .+ exp.(-params4))
    
    # Transform beta parameter
    #Beta = 50 ./ (1 .+ exp.(-params5))
        Beta = 25 .+ 25 .* erf.(params5 ./ sqrt(2))

    
    # Return DataFrame with transformed parameters
    return DataFrame(
        Conf_True = Conf_True ,
        Disconf_True = Disconf_True,
        Conf_False = Conf_False,
        Disconf_False = Disconf_False,
        Beta = Beta
    )
end


# Calculate SEM directly from individual subject data
betas_matrix = Matrix(reshape(betas, 1, length(betas)))
se_matrix = Matrix(se)

# Calculate 95% Confidence Interval bounds
xh_95ci = betas .+ 1.96 .* se
xl_95ci = betas .- 1.96 .* se

# Calculate SEM from CI
sem = (xh_95ci - xl_95ci) ./ (2 * 1.96)

# Transform the point estimates, CI bounds, and SEM bounds
df_point = transform_back(betas_matrix)
df_ci_upper = transform_back(xh_95ci)
df_ci_lower = transform_back(xl_95ci)
df_sem_upper = transform_back(betas_matrix .+ sem)
df_sem_lower = transform_back(betas_matrix .- sem)

# Create the final DataFrame
final_df = DataFrame()
for col in names(df_point)
    final_df[!, "$(col)_point"] = df_point[!, col]
    final_df[!, "$(col)_ci_lower"] = df_ci_lower[!, col]
    final_df[!, "$(col)_ci_upper"] = df_ci_upper[!, col]
    final_df[!, "$(col)_sem_lower"] = df_sem_lower[!, col]
    final_df[!, "$(col)_sem_upper"] = df_sem_upper[!, col]
end

# Display the final DataFrame
println(final_df)


## Save the final DataFrame to a CSV file

In [ ]:

CSV.write("M4StatsS2.csv", final_df)

## Use the same function to create the estimates for each subject 

In [ ]:
df = transform_back(x)


## write it to a csv

In [ ]:
CSV.write("M4PointsS2.csv", df)